# Creacion de Ventanas Deslizantes con metodo por Transecto y metodo general

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Creación de ventanas deslizantes: 72h entrada, 72h salida.
Genera arrays X_ml (2D aplanado), X_dl (3D), y (72h).
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")
WINDOWS_DIR = os.path.join(BASE_DIR, "windows")

INPUT_ML_TRANSECT = os.path.join(ENCODED_DIR, "ml", "by_transect")
INPUT_DL_TRANSECT = os.path.join(ENCODED_DIR, "dl", "by_transect")
INPUT_ML_GLOBAL = os.path.join(ENCODED_DIR, "ml", "global")
INPUT_DL_GLOBAL = os.path.join(ENCODED_DIR, "dl", "global")

OUTPUT_ML_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(WINDOWS_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(WINDOWS_DIR, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = "O3"


def standardize_target_column(df):
    df = df.copy()
    if "O3" not in df.columns and "O3_for_impute" in df.columns:
        df = df.rename(columns={"O3_for_impute": "O3"})
    elif "O3" in df.columns and "O3_for_impute" in df.columns:
        df["O3"] = df["O3"].where(df["O3"].notna(), df["O3_for_impute"])
        df = df.drop(columns=["O3_for_impute"])
    return df


def ensure_datetime_index(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.loc[~df.index.isna()].copy()
    return df


def select_numeric_features(df):
    df = df.copy()
    df = df.select_dtypes(include=[np.number, "bool"]).copy()
    if df.shape[1] > 0:
        df = df.apply(pd.to_numeric, errors="coerce")
    return df


def create_windows_with_timestamps(df, target_col=TARGET_COL, window_in=WINDOW_IN, window_out=WINDOW_OUT):
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame debe tener índice DatetimeIndex")
    df_sorted = df.sort_index(kind="mergesort")
    data = df_sorted.values
    feature_names = df_sorted.columns.tolist()
    if target_col not in df_sorted.columns:
        raise KeyError(f"No existe columna '{target_col}'")
    n = len(df_sorted)
    X_ml_list = []
    X_dl_list = []
    y_list = []
    timestamps = []
    last_i = n - window_out
    if last_i < window_in:
        return None, None, None, None, feature_names
    for i in range(window_in, last_i + 1):
        in_data = data[i - window_in:i, :]          # (window_in, n_features)
        out_data = df_sorted[target_col].iloc[i:i + window_out].values
        X_dl_list.append(in_data)
        X_ml_list.append(in_data.flatten())
        y_list.append(out_data)
        timestamps.append(df_sorted.index[i])
    if len(X_ml_list) == 0:
        return None, None, None, None, feature_names
    X_ml = np.array(X_ml_list, dtype=np.float32)
    X_dl = np.array(X_dl_list, dtype=np.float32)   # ya es 3D
    y = np.array(y_list, dtype=np.float32)
    timestamps = np.array(timestamps, dtype="datetime64[h]")
    return X_ml, X_dl, y, timestamps, feature_names


def process_file_ml_dl(ml_path, dl_path, output_ml_dir, output_dl_dir, name):
    print(f"  Procesando {name}...")
    df_ml = pd.read_csv(ml_path, index_col=0, parse_dates=True, low_memory=False)
    df_dl = pd.read_csv(dl_path, index_col=0, parse_dates=True, low_memory=False)
    df_ml = ensure_datetime_index(df_ml)
    df_dl = ensure_datetime_index(df_dl)
    df_ml = standardize_target_column(df_ml)
    df_dl = standardize_target_column(df_dl)
    if not df_ml.index.equals(df_dl.index):
        common_index = df_ml.index[df_ml.index.isin(df_dl.index)]
        if len(common_index) == 0:
            print(f"    Error: no hay timestamps comunes.")
            return
        df_ml = df_ml.loc[common_index].copy()
        df_dl = df_dl.loc[common_index].copy()
    df_ml = select_numeric_features(df_ml)
    df_dl = select_numeric_features(df_dl)
    if df_ml.shape[1] == 0 or df_dl.shape[1] == 0:
        print(f"    Error: sin características numéricas.")
        return
    result_ml = create_windows_with_timestamps(df_ml)
    if result_ml[0] is None:
        print(f"    No se generaron ventanas ML.")
        return
    X_ml, _, y, timestamps, feat_names_ml = result_ml
    result_dl = create_windows_with_timestamps(df_dl)
    if result_dl[0] is None:
        print(f"    No se generaron ventanas DL.")
        return
    X_dl, _, _, _, _ = result_dl
    if len(X_ml) != len(X_dl):
        print(f"    Error: número de ventanas difiere.")
        return
    # Asegurar que X_dl sea 3D (por si acaso)
    if X_dl.ndim == 2:
        n_samples = X_dl.shape[0]
        n_features = X_dl.shape[1] // WINDOW_IN
        X_dl = X_dl.reshape(n_samples, WINDOW_IN, n_features)
        print(f"    Remodelado X_dl a {X_dl.shape}")
    np.save(os.path.join(output_ml_dir, f"{name}_X.npy"), X_ml)
    np.save(os.path.join(output_ml_dir, f"{name}_y.npy"), y)
    np.save(os.path.join(output_ml_dir, f"{name}_timestamps.npy"), timestamps)
    np.save(os.path.join(output_dl_dir, f"{name}_X.npy"), X_dl)
    np.save(os.path.join(output_dl_dir, f"{name}_y.npy"), y)
    np.save(os.path.join(output_dl_dir, f"{name}_timestamps.npy"), timestamps)
    print(f"    Ventanas guardadas: {len(X_ml)} muestras. Shapes: X_ml {X_ml.shape}, X_dl {X_dl.shape}")


def process_by_transect():
    print("\n--- Procesando datos por transecto ---")
    if not os.path.exists(INPUT_ML_TRANSECT) or not os.path.exists(INPUT_DL_TRANSECT):
        print("Carpetas de entrada no existen.")
        return
    ml_files = {f.stem: f for f in Path(INPUT_ML_TRANSECT).glob("*.csv")}
    dl_files = {f.stem: f for f in Path(INPUT_DL_TRANSECT).glob("*.csv")}
    common = set(ml_files.keys()) & set(dl_files.keys())
    for name in sorted(common):
        process_file_ml_dl(ml_files[name], dl_files[name], OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, name)


def process_global():
    print("\n--- Procesando datos globales (por estación) ---")
    if not os.path.exists(INPUT_ML_GLOBAL) or not os.path.exists(INPUT_DL_GLOBAL):
        print("Carpetas de entrada no existen.")
        return
    ml_files = {f.stem: f for f in Path(INPUT_ML_GLOBAL).glob("*.csv")}
    dl_files = {f.stem: f for f in Path(INPUT_DL_GLOBAL).glob("*.csv")}
    common = set(ml_files.keys()) & set(dl_files.keys())
    for name in sorted(common):
        process_file_ml_dl(ml_files[name], dl_files[name], OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, name)


if __name__ == "__main__":
    print("Creación de ventanas deslizantes (72h in / 72h out)")
    process_by_transect()
    process_global()
    print("Proceso completado.")

Creación de ventanas deslizantes (72h in / 72h out)

--- Procesando datos por transecto ---
  Procesando Transecto_1...
    Remodelado X_dl a (194267, 72, 23)
    Ventanas guardadas: 194267 muestras. Shapes: X_ml (194267, 1728), X_dl (194267, 72, 23)
  Procesando Transecto_2...
    Remodelado X_dl a (183256, 72, 23)
    Ventanas guardadas: 183256 muestras. Shapes: X_ml (183256, 1728), X_dl (183256, 72, 23)
  Procesando Transecto_3...
    Remodelado X_dl a (236980, 72, 23)
    Ventanas guardadas: 236980 muestras. Shapes: X_ml (236980, 1728), X_dl (236980, 72, 23)
  Procesando Transecto_4...
    Remodelado X_dl a (247969, 72, 23)
    Ventanas guardadas: 247969 muestras. Shapes: X_ml (247969, 1728), X_dl (247969, 72, 23)
  Procesando Transecto_6...
    Remodelado X_dl a (319015, 72, 23)
    Ventanas guardadas: 319015 muestras. Shapes: X_ml (319015, 1728), X_dl (319015, 72, 23)
  Procesando Transecto_7...
    Remodelado X_dl a (359056, 72, 23)
    Ventanas guardadas: 359056 muestras. Shape